# Notebook 15 – Imbalanced Image Classification



In [1]:
import time
import numpy as np
import torch
from torch import nn
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

PyTorch: 2.14.0+cpu | device: cpu


## Concept and dataset example
If 95% of images belong to one class, always predicting that class gives 95% accuracy while missing every rare case. Weighted loss penalizes rare-class errors more. Balanced sampling changes which examples appear per batch. Oversampling can overfit duplicates; augmentation adds variation only when transforms preserve labels. Compare macro-F1, per-class recall, and confusion matrix alongside accuracy.

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,f1_score,classification_report
from collections import Counter
d=load_digits(); labels=d.target
keep=np.concatenate([np.flatnonzero(labels==c)[:(700 if c==0 else 15)] for c in range(10)])
x=d.data[keep]/16; y=labels[keep]; xtr,xte,ytr,yte=train_test_split(x,y,test_size=.3,random_state=12,stratify=y)
print('train distribution:',Counter(ytr))
for name,weights in [('unweighted',None),('class_weight=balanced','balanced')]:
 model=LogisticRegression(max_iter=1000,class_weight=weights).fit(xtr,ytr); pred=model.predict(xte)
 print(name,'accuracy',round(accuracy_score(yte,pred),3),'macro_f1',round(f1_score(yte,pred,average='macro'),3)); print(classification_report(yte,pred,digits=3,zero_division=0))

train distribution: Counter({np.int64(0): 125, np.int64(9): 11, np.int64(5): 11, np.int64(8): 11, np.int64(1): 11, np.int64(7): 10, np.int64(3): 10, np.int64(6): 10, np.int64(4): 10, np.int64(2): 10})
unweighted accuracy 0.979 macro_f1 0.96
              precision    recall  f1-score   support

           0      0.981     1.000     0.991        53
           1      1.000     1.000     1.000         4
           2      1.000     1.000     1.000         5
           3      1.000     1.000     1.000         5
           4      1.000     1.000     1.000         5
           5      1.000     0.750     0.857         4
           6      1.000     1.000     1.000         5
           7      1.000     1.000     1.000         5
           8      1.000     1.000     1.000         4
           9      0.750     0.750     0.750         4

    accuracy                          0.979        94
   macro avg      0.973     0.950     0.960        94
weighted avg      0.979     0.979     0.978        94



## PyTorch options, results, interpretation
For a CNN, compute weights from training counts only, e.g. `total / (num_classes * class_count)`, and pass them to `CrossEntropyLoss(weight=...)`. For sampling, give each example weight `1 / class_count[label]` to `WeightedRandomSampler(replacement=True)`. Avoid combining aggressive weights and sampling without validation; that can over-correct.

| Method | Accuracy | Macro-F1 | Per-class recall | Time |
|---|---:|---:|---|---:|
| Unweighted | measure | measure | measure | measure |
| Weighted loss | measure | measure | measure | measure |
| Balanced sampler | measure | measure | measure | measure |

